In [1]:
!pip install -q "numpy<2.0" insightface onnxruntime-gpu onnxruntime albumentations


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 15.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 93.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 37.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.4 MB/s eta 0:00:00


In [2]:
'''

pip uninstall torch torchvision transformers -y


'''

'\n\npip uninstall torch torchvision transformers -y\n\n\n'

In [3]:
pip install --upgrade torch torchvision transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 22.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.0 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 24.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 7.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import pandas as pd
import re
from collections import Counter



In [5]:
# ==========================================
# POLIMEME DECODE v3.0: INFERENCE ONLY
# (Pure NLP with Dual-Context Processing)
# ==========================================
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# ================= CONFIGURATION =================

# ================= CONFIGURATION =================
INFERENCE_CONFIG = {
    # Data paths
    'test_csv': '/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Test/Test.csv',
    'model_dir': '/kaggle/input/kfold-models/outputs',  # ✓ UPDATED PATH
    'output_dir': '/kaggle/working/outputs',
    
    # Context paths
    'test_context_csv': '/kaggle/input/cleaned-text/cleaned_meme_text_data.csv',
    
    # Model names (must match training config)
    'primary_model': 'roberta-base',
    'secondary_model': 'distilroberta-base',
    
    # Model Strategy
    'use_dual_encoder': True,
    'context_weight': 3.0,
    'ocr_weight': 1.5,
    
    # Inference Parameters
    'num_folds': 5,  # Will load text_classifier_fold0.pth through fold4.pth
    'batch_size': 8,
    'max_len_primary': 256,
    'max_len_secondary': 192,
    'num_classes': 2,
    'mixed_precision': True,
    
    # Device
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
}

# Label Mappings
ID2LABEL = {0: 'NonPolitical', 1: 'Political'}

# ==========================================
# LOAD TEST DATA
# ==========================================
def load_test_data():
    """Load and prepare test data with OCR and context"""
    
    df_test = pd.read_csv(INFERENCE_CONFIG['test_csv'])
    
    # Merge OCR data
    try:
        df_test_ocr = pd.read_csv('/kaggle/input/results-ocr/cuet_cse_test_text_hunyuanocr.csv')
        df_test = pd.merge(df_test, df_test_ocr, on='Image_name', how='left')
        print("✓ OCR data merged successfully")
    except FileNotFoundError:
        print("⚠ OCR files not found, proceeding without OCR merge.")
        df_test['text'] = ''
    
    # Merge context data
    if INFERENCE_CONFIG['test_context_csv'] and os.path.exists(INFERENCE_CONFIG['test_context_csv']):
        df_context_test = pd.read_csv(INFERENCE_CONFIG['test_context_csv'])
        df_test = pd.merge(df_test, df_context_test, on='Image_name', how='left', suffixes=('', '_ctx'))
        print("✓ Test context loaded successfully")
    else:
        print("⚠ Test context not available - using empty strings")
        df_test['combined_meaning'] = ''
        df_test['context_analysis'] = ''
    
    # Fill NaN values
    text_cols = ['text', 'combined_meaning', 'context_analysis']
    for col in text_cols:
        if col in df_test.columns:
            df_test[col] = df_test[col].fillna('').astype(str)
        else:
            df_test[col] = ''

    print(f"✓ Test shape: {df_test.shape}")
    
    return df_test

# ==========================================
# DUAL-ENCODER TEXT CLASSIFIER (Same as training)
# ==========================================
class DualEncoderTextClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        
        # PRIMARY ENCODER
        self.primary_encoder = AutoModel.from_pretrained(INFERENCE_CONFIG['primary_model'])
        primary_dim = self.primary_encoder.config.hidden_size
        
        # SECONDARY ENCODER
        if INFERENCE_CONFIG['use_dual_encoder']:
            self.secondary_encoder = AutoModel.from_pretrained(INFERENCE_CONFIG['secondary_model'])
            secondary_dim = self.secondary_encoder.config.hidden_size
        else:
            self.secondary_encoder = None
            secondary_dim = 0
        
        # OCR TEXT PROCESSING
        self.ocr_projector = nn.Sequential(
            nn.Linear(primary_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # CONTEXT PROCESSING (Combined Meaning)
        self.context_combined_projector = nn.Sequential(
            nn.Linear(primary_dim, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.LayerNorm(512),
            nn.GELU()
        )
        
        # CONTEXT PROCESSING (Context Analysis)
        if INFERENCE_CONFIG['use_dual_encoder']:
            self.context_analysis_projector = nn.Sequential(
                nn.Linear(secondary_dim, 768),
                nn.LayerNorm(768),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(768, 512),
                nn.LayerNorm(512),
                nn.GELU()
            )
        else:
            self.context_analysis_projector = nn.Sequential(
                nn.Linear(primary_dim, 768),
                nn.LayerNorm(768),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(768, 512),
                nn.LayerNorm(512),
                nn.GELU()
            )
        
        # CROSS-CONTEXT ATTENTION
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=512,
            num_heads=8,
            dropout=0.2,
            batch_first=True
        )
        
        # FEATURE FUSION
        fusion_dim = 512 + 512 + 512
        
        self.fusion_gate = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim),
            nn.Sigmoid()
        )
        
        # CLASSIFIER HEAD
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.5),
            
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.4),
            
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, num_classes)
        )
        
    def forward(self, ocr_input, combined_input, analysis_input):
        # OCR TEXT ENCODING
        ocr_output = self.primary_encoder(
            input_ids=ocr_input['ids'], 
            attention_mask=ocr_input['mask']
        )
        ocr_emb = ocr_output.last_hidden_state[:, 0, :]
        ocr_feat = self.ocr_projector(ocr_emb) * INFERENCE_CONFIG['ocr_weight']
        
        # COMBINED MEANING ENCODING
        combined_output = self.primary_encoder(
            input_ids=combined_input['ids'],
            attention_mask=combined_input['mask']
        )
        combined_emb = combined_output.last_hidden_state[:, 0, :]
        combined_feat = self.context_combined_projector(combined_emb) * INFERENCE_CONFIG['context_weight']
        
        # CONTEXT ANALYSIS ENCODING
        if INFERENCE_CONFIG['use_dual_encoder']:
            analysis_output = self.secondary_encoder(
                input_ids=analysis_input['ids'],
                attention_mask=analysis_input['mask']
            )
        else:
            analysis_output = self.primary_encoder(
                input_ids=analysis_input['ids'],
                attention_mask=analysis_input['mask']
            )
        
        analysis_emb = analysis_output.last_hidden_state[:, 0, :]
        analysis_feat = self.context_analysis_projector(analysis_emb) * INFERENCE_CONFIG['context_weight']
        
        # CROSS-ATTENTION
        combined_attn = combined_feat.unsqueeze(1)
        analysis_attn = analysis_feat.unsqueeze(1)
        
        attended_combined, _ = self.cross_attention(combined_attn, analysis_attn, analysis_attn)
        attended_combined = attended_combined.squeeze(1)
        
        attended_analysis, _ = self.cross_attention(analysis_attn, combined_attn, combined_attn)
        attended_analysis = attended_analysis.squeeze(1)
        
        # FEATURE FUSION
        all_features = torch.cat([ocr_feat, attended_combined, attended_analysis], dim=1)
        gate = self.fusion_gate(all_features)
        gated_features = all_features * gate
        
        # CLASSIFICATION
        logits = self.classifier(gated_features)
        
        return logits

# ==========================================
# DATASET
# ==========================================
class TextOnlyDataset(Dataset):
    def __init__(self, df, primary_tokenizer, secondary_tokenizer):
        self.df = df.reset_index(drop=True)
        self.primary_tokenizer = primary_tokenizer
        self.secondary_tokenizer = secondary_tokenizer

    def __len__(self): 
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Tokenize OCR text
        ocr_text = str(row['text']) if len(str(row['text'])) > 1 else "unclear text"
        ocr_tok = self.primary_tokenizer(
            ocr_text,
            padding='max_length',
            truncation=True,
            max_length=INFERENCE_CONFIG['max_len_secondary'],
            return_tensors='pt'
        )
        
        # Tokenize combined meaning
        combined_tok = self.primary_tokenizer(
            str(row['combined_meaning']),
            padding='max_length',
            truncation=True,
            max_length=INFERENCE_CONFIG['max_len_primary'],
            return_tensors='pt'
        )
        
        # Tokenize context analysis
        if INFERENCE_CONFIG['use_dual_encoder']:
            analysis_tok = self.secondary_tokenizer(
                str(row['context_analysis']),
                padding='max_length',
                truncation=True,
                max_length=INFERENCE_CONFIG['max_len_primary'],
                return_tensors='pt'
            )
        else:
            analysis_tok = self.primary_tokenizer(
                str(row['context_analysis']),
                padding='max_length',
                truncation=True,
                max_length=INFERENCE_CONFIG['max_len_primary'],
                return_tensors='pt'
            )
        
        item = {
            'ocr_ids': ocr_tok['input_ids'].squeeze(0),
            'ocr_mask': ocr_tok['attention_mask'].squeeze(0),
            'combined_ids': combined_tok['input_ids'].squeeze(0),
            'combined_mask': combined_tok['attention_mask'].squeeze(0),
            'analysis_ids': analysis_tok['input_ids'].squeeze(0),
            'analysis_mask': analysis_tok['attention_mask'].squeeze(0),
        }
            
        return item

# ==========================================
# INFERENCE
# ==========================================
def generate_submission():
    print("="*50)
    print("INFERENCE MODE - DUAL-ENCODER TEXT CLASSIFIER")
    print("="*50)
    
    df_test = load_test_data()
    
    # Load tokenizers
    print("\nLoading tokenizers...")
    primary_tokenizer = AutoTokenizer.from_pretrained(INFERENCE_CONFIG['primary_model'])
    secondary_tokenizer = AutoTokenizer.from_pretrained(INFERENCE_CONFIG['secondary_model']) if INFERENCE_CONFIG['use_dual_encoder'] else primary_tokenizer
    
    # Create dataset and loader
    test_dataset = TextOnlyDataset(df_test, primary_tokenizer, secondary_tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=INFERENCE_CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    
    # Initialize probabilities array
    avg_probs = np.zeros((len(df_test), INFERENCE_CONFIG['num_classes']))
    
    # Load and run inference for each fold
    for fold in range(INFERENCE_CONFIG['num_folds']):
        model_path = f"{INFERENCE_CONFIG['model_dir']}/text_classifier_fold{fold}.pth"
        
        if not os.path.exists(model_path): 
            print(f"⚠ Fold {fold} model not found at {model_path}")
            continue
        
        print(f"\nLoading Fold {fold} model...")
        model = DualEncoderTextClassifier(num_classes=INFERENCE_CONFIG['num_classes']).to(INFERENCE_CONFIG['device'])
        model.load_state_dict(torch.load(model_path, map_location=INFERENCE_CONFIG['device']))
        model.eval()
        
        fold_probs = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Fold {fold} Inference"):
                ocr_input = {
                    'ids': batch['ocr_ids'].to(INFERENCE_CONFIG['device']),
                    'mask': batch['ocr_mask'].to(INFERENCE_CONFIG['device'])
                }
                
                combined_input = {
                    'ids': batch['combined_ids'].to(INFERENCE_CONFIG['device']),
                    'mask': batch['combined_mask'].to(INFERENCE_CONFIG['device'])
                }
                
                analysis_input = {
                    'ids': batch['analysis_ids'].to(INFERENCE_CONFIG['device']),
                    'mask': batch['analysis_mask'].to(INFERENCE_CONFIG['device'])
                }
                
                if INFERENCE_CONFIG['mixed_precision']:
                    with torch.cuda.amp.autocast():
                        logits = model(ocr_input, combined_input, analysis_input)
                else:
                    logits = model(ocr_input, combined_input, analysis_input)
                    
                fold_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        
        avg_probs += np.array(fold_probs)
        
        del model
        torch.cuda.empty_cache()
        print(f"✓ Fold {fold} completed")
    
    # Average predictions across folds
    avg_probs /= INFERENCE_CONFIG['num_folds']
    
    # Create submission
    submission = pd.DataFrame({
        'Image_name': df_test['Image_name'], 
        'Target': [ID2LABEL[p] for p in np.argmax(avg_probs, axis=1)]
    })
    
    output_path = f"{INFERENCE_CONFIG['output_dir']}/submission_inference_only.csv"
    submission.to_csv(output_path, index=False)
    
    print(f"\n{'='*50}")
    print(f"✓ Submission saved to: {output_path}")
    print(f"Total predictions: {len(submission)}")
    print(f"Class distribution:")
    print(submission['Target'].value_counts())
    print(f"{'='*50}")

# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    os.makedirs(INFERENCE_CONFIG['output_dir'], exist_ok=True)
    generate_submission()

INFERENCE MODE - DUAL-ENCODER TEXT CLASSIFIER
✓ OCR data merged successfully
✓ Test context loaded successfully
✓ Test shape: (330, 6)

Loading tokenizers...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

✓ Fold 2 completed

Loading Fold 3 model...


Fold 3 Inference: 100%|██████████| 42/42 [00:02<00:00, 15.38it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Fold 3 completed

Loading Fold 4 model...


Fold 4 Inference: 100%|██████████| 42/42 [00:02<00:00, 15.43it/s]


✓ Fold 4 completed

✓ Submission saved to: /kaggle/working/outputs/submission_inference_only.csv
Total predictions: 330
Class distribution:
Target
NonPolitical    169
Political       161
Name: count, dtype: int64


In [13]:
df1=pd.read_csv('/kaggle/input/test-4-texts/cuet_csv_combined_meaning_on_test.csv')
df2=pd.read_csv('/kaggle/input/test-4-texts/cuet_csv_context_analysis_on_test.csv')
df3=pd.read_csv('/kaggle/input/test-4-texts/cuet_csv_image_description_on_test.csv')
df4=pd.read_csv('/kaggle/input/test-4-texts/cuet_csv_text_meaning_on_test.csv')

from functools import reduce
import pandas as pd

# 1. Put all your DataFrames into a list
dfs = [df1, df2, df3, df4]

# 2. Merge them sequentially
# how='outer': Keeps all images. If an image is missing in one file, it puts NaN (Safer option).
# how='inner': Keeps only images that appear in ALL 6 files.
df= reduce(lambda left, right: pd.merge(left, right, on='Image_name', how='outer'), dfs)

# 3. Check the shape and preview
print(f"Final shape: {df.shape}")


# Check the result
df.head()


Final shape: (330, 5)


,Image_name,combined_meaning,context_analysis,image_description,text_meaning
0,test0001.jpg,The meme intends to humorously portray disappo...,This meme is likely used to humorously portray...,The meme shows a young man with dark hair and ...,This meme uses a deliberately misspelled phras...
1,test0002.jpg,The meme likely intends to express confusion o...,This meme likely originated from a moment in a...,The meme features a young man with curly brown...,"""AFC kya matlab iqe fue"" is a phrase heavily a..."
2,test0003.jpg,The meme uses the image of Tom from Tom and Je...,This meme likely references a situation where ...,The meme features the cartoon character Tom fr...,"This meme uses the word ""SNACKS"" ironically, l..."
3,test0004.jpg,"The meme uses the image of the Hulk, a symbol ...",This meme is used to express a feeling of bein...,The meme features two images of the Hulk. The ...,The text expresses extreme disbelief and amuse...
4,test0005.jpg,The meme conveys a sense of **disappointment a...,This meme is used to express feeling financial...,The meme features a medium shot of a man with ...,This meme text expresses a feeling of being le...


In [14]:
import pandas as pd
import re

# ==========================================
# 1. Define Dictionaries
# ==========================================

political_vocabulary = {
    "parties_and_organizations": [
        "bjp", "বিজেপি", "trinamool", "tmc", "aitc", "তৃণমূল", "তুণমূল",
        "congress", "inc", "কংগ্রেস", "cpm", "cpim", "communist", "marxist", 
        "leftist", "সিপিএম", "বাম",
        "awami", "awami league", 
        "আওয়ামী", "আওয়ামী লীগ", "লীগ", "bnp", "বিএনপি", "rss", "আরএসএস", 
        "shibir", "শিবির", "chhatra league", "ছাত্রলীগ", "police", "rab", 
        "পুলিশ", "র‍্যাব", "দল"
    ],
    "political_figures": [
        "modi", "narendra", "narendra modi", "মোদী", "নরেন্দ্র মোদী",
        "mamata", "banerjee", "mamata banerjee", "mamatadi", "মমতা", 
        "বন্দ্যোপাধ্যায়", "hasina", "sheikh hasina", "হাসিনা", 
        "শেখ হাসিনা", "khaleda", "zia", "khaleda zia", "খালেদা", "জিয়া", 
        "mujib", "mujibur", "bangabandhu", "মুজিব", "বঙ্গবন্ধু", "abhishek", 
        "abhishek banerjee", "অভিষেক", "suvendu", "adhikari", "suvendu adhikari", 
        "শুভেন্দু", "partha", "chatterjee", "partha chatterjee", "পার্থ", 
        "anubrata", "mondal", "anubrata mondal", "keshto", "অনুব্রত", "মন্ডল", 
        "কেষ্ট", "kunal", "ghosh", "kunal ghosh", "কুণাল", "firhad", "hakim", 
        "bobby", "ফিরহাদ", "হাকিম", "biplob", "biplob deb", "বিপ্লব", "dilip", 
        "ghosh", "দিলীপ", "gandhi", "rahul", "রাহুল", "গান্ধী",'নয়ন',"জাকির",'Elias Hossain'
    ],
    "slang_and_codenames": [
        "pisi", "pishi", "পিসি", "bhaipo", "ভাইপো", "chor", "thief", "চোর", 
        "dalal", "agent", "দালাল", "didi", "didiro", "দিদি", "দিদির",'কটর', 
        "neta", "leader", "নেতা", "chamcha", "sycophant", "চামচা", "bhakt", 
        "bhakts", "ভক্ত", "অন্ধভক্ত", "choti",'ROG'
    ],
    "key_issues_and_events": [
        "scam", "দুর্নীতি", "স্ক্যাম", "কেলেঙ্কারি", 
        "nishi", "coal scam", 
        "কয়লা পাচার", "smuggling", 
        "গরু পাচার", "protest", "movement", "dharna", "আন্দোলন", 
        "বিক্ষোভ", "ধরণা", "strike", "hartal", "bandh", "হরতাল",
        "ধর্মঘট", "election", "vote", "nirbachon", "নির্বাচন", "ভোট", 
        "democracy", "democratic", "ganatantra", "গণতন্ত্র", "fascism", 
        "fascist", "ফ্যাসিবাদ", "ফ্যাসিস্ট", "স্বৈরাচার", 
        "unnayan", "উন্নয়ন", "khela hobe", "খেলা হবে", 'সজন'
         "inflation", "price hike", "মূল্যবৃদ্ধি","সজন হারানোর বেদনা",'হাসি না',"Nila Market"
    ],
    "agencies_and_governance": [
        "government", "govt", "sorkar", "সরকার", "minister", "montri", 
        "मंत्री", "ministry", "montronaloy", "মন্ত্রণালয়", "cabinet", 
        "ক্যাবিনেট", "মন্ত্রিসভা", "opposition", "birodhi", "বিরোধী",
        "enforcement directorate","cbi", "সিবিআই", "court", "high court", 
        "supreme court", "আদালত", "কোর্ট", "হাইকোর্ট", "justice", "bichar", 
        "বিচার", "jail", "prison", "hajot", "জেল", "কারাগার", "হাজত", "bail", 
        "jamini", "জামিন"
    ],
    "ideology_and_identity": [
        "secular", "secularism", "dharmaniropekkho", "ধর্মনিরপেক্ষ", "communal", 
        "sampradayik", "সাম্প্রদায়িক",  "hindutva", 
        "হিন্দুত্ব", "muslim", "islam", "মুসলিম", "ইসলাম", "nationalist", 
        "deshpremik", "জাতীয়তাবাদী", "দেশপ্রেমিক", "freedom", 
        "swadhinata", "azadi", "স্বাধীনতা", "আজাদী", 
        "jonogon", "janata", "জনগণ", "জনতা", "rashtra", "provinces", 
        "রাষ্ট্র", "রাজ্য"
    ]
}

political_vocabulary_v2 = {
    "parties_and_organizations": [
        "awami", "awami league", "আওয়ামী", "আওয়ামী লীগ", "লীগ",
        "bnp", "bangladesh nationalist party", "বিএনপি",
        "jamaat", "jamaateislami", "jamaat-e-islami", "জামাত", "জামায়াত",
        "shibir", "student shibir", "শিবির",
        "chhatra league", "bcl", "chatra league", "ছাত্রলীগ",
        "chhatra dal", "chatrodol", "jcd", "ছাত্রদল",
        "jubo league", "jubo", "যুবলীগ",
        "bjp", "বিজেপি",
        "congress", "kangeres", "কংগ্রেস",
        "police", "rab", "পুলিশ", "র‍্যাব", "সেনাবাহিনী",
        "un", "united nations", "জাতিসংঘ"
    ],
    "political_figures": [
        "hasina", "sheikh hasina", "hasina", "হাসিনা", "শেখ হাসিনা",
        "khaleda", "zia", "khaleda zia", "begum zia", "খালেদা", "জিয়া", "খালেদা জিয়া",
        "mujib", "mujibur", "rahman", "bangabandhu", "মুজিব", "বঙ্গবন্ধু", "শেখ মুজিব",
        "tareq", "tarique", "rahman", "tarek", "তারেক", "তারেক জিয়া",
        "yunus", "muhammad yunus", "dr yunus", "ইউনুস", "ডঃ ইউনুস",
        "quader", "obaidul", "কাদের", "ওবায়দুল",
        "modi", "narendra", "narendra modi", "মোদী", "নরেন্দ্র মোদী",
        "mamata", "banerjee", "mamata banerjee", "mamatadi", "মমতা",
        "donald trump", "ট্রাম্প",
        "putin", "পুতিন",
        "hitler", "হিটলার",
        # NEW ADDITIONS
        "rumin", "farhana", "rumin farhana", # Rumin Farhana
        "ghulam", "golam", "azam", "golam azam", "গোলাম", "আযম", "গোলাম আযম", # Ghulam Azam
        "saka", "chowdhury", "saka chowdhury", "সাকা", "চৌধুরী", "সাকা চৌধুরী", # Saka Chowdhury
        "sayeedi", "sayedee", "delwar hossain sayeedi", "সাঈদী", "দেলওয়ার হোসাইন সাঈদী", # Sayeedi
        "maulana", "mawlana", "মাওলানা" # Title
    ],
    "slang_and_codenames": [
        "vote chor", "votechor", "ভোটচোর",
        "agent",  "দালাল", "ভারতীয় দালাল",
        "razakar", "rajakar", "রাজাকার",
        "shahbaghi", "shahbag", "শাহবাগী", "শাহবাগ",
        "nastik", "atheist", "নাস্তিক",
        "jongi", "militant", "terrorist", "জঙ্গি",
        "chamcha", "চামচা", "পা চাটা",
        "bot", "paid bot", "বট",
        "pisi", "pishi", "পিসি",
        # NEW ADDITION
        "patowari", "PATOWARI" # Slang
    ],
    "key_issues_and_events": [
        "durniti", "durobhinga", "দুর্নীতি", "লুটপাট",
        "election", "nirbachon", "vote", "poll", "নির্বাচন", "ভোট", "কারচুপি",
        "protest", "andolon", "movement", "dharna", "আন্দোলন", "বিক্ষোভ", "গণজাগরণ",
        "strike", "hartal", "oborodh", "blockade", "হরতাল", "অবরোধ", "ধর্মঘট",
        "quota", "quota reform", "kota", "কোটা", "কোটা সংস্কার",
        "reform", "songskar", "সংস্কার",
        "liberation war", "muktijuddho", "1971", "মুক্তিযুদ্ধ", "একাত্তর", "স্বাধীনতা",
        "genocide", "gonohotta", "গণহত্যা",
        "democracy", "gonotontro", "গণতন্ত্র",
        "fascism", "fascist", "swairachar", "dictator", "ফ্যাসিবাদ", "ফ্যাসিস্ট", "স্বৈরাচার",
        "rights", "odhikar", "human rights", "অধিকার", "মানবাধিকার",
        "inflation", "price hike", "মূল্যবৃদ্ধি"
    ],
    "governance_and_structures": [
        "government", "sorkar", "govt", "regime", "সরকার", "শাসন",
        "opposition", "birodhi", "birodhi dol", "বিরোধী", "বিরোধী দল",
        "parliament", "sangsad", "sangsad bhaban", "সংসদ", "পার্লামেন্ট",
        "ministry", "montronaloy", "মন্ত্রণালয়",
        "cabinet", "montrisobha", "মন্ত্রিসভা",
        "constitution", "shongbidhan", "সংবিধান",
        "court", "adalat", "high court", "supreme court", "আদালত", "কোর্ট", "হাইকোর্ট",
        "justice", "bichar", "বিচার",
        "administration", "proshason", "প্রশাসন"
    ],
    # NEW CATEGORIES ADDED
    "roles_and_attributes": [
        "candidate", "podoprarthi", "পদপ্রার্থী",
        "seat", "constituency", "asoner", "আসনের",
        "supporter", "sportsman", "সাপোর্টার",
        "distinguished", "bishishtho", "বিশিষ্ট",
        "hardline", "kottor", "কট্টর",
        "rightist", "right-wing", "danponthi", "ডানপন্থী"
    ],
    "entities_and_locations": [
        "bangladesh", "বাংলাদেশ"
    ]
}
# ==========================================
# 2. Combine and Compile Logic
# ==========================================

# Create a single set to hold unique words from BOTH dictionaries
all_keywords = set()

# Add words from the first dictionary
for category_words in political_vocabulary.values():
    for word in category_words:
        all_keywords.add(str(word).lower())

# Add words from the second dictionary (Fix: This part was missing in your logic)
for category_words in political_vocabulary_v2.values():
    for word in category_words:
        all_keywords.add(str(word).lower())

# Sort by length (descending) to match compound words first
sorted_keywords = sorted(list(all_keywords), key=len, reverse=True)

# Compile Regex
pattern = r'\b(?:' + '|'.join(map(re.escape, sorted_keywords)) + r')\b'
compiled_regex = re.compile(pattern, re.IGNORECASE | re.UNICODE)

# ==========================================
# 3. Search Function
# ==========================================

def find_political_context(text):
    """
    Returns 1 if any political keyword is found, else 0.
    Handles NaN/Null values gracefully.
    """
    if pd.isna(text) or text == '':
        return 0
    
    # Ensure text is string and convert to lowercase
    text_str = str(text).lower()
    
    if compiled_regex.search(text_str):
        return 1
    return 0

# ==========================================
# 4. Apply to DataFrame
# ==========================================

# Replace 'df' with your actual dataframe name
# This applies the function to the 'text_meaning' column as requested

if 'text_meaning' in df.columns:
    df['is_political_keyword'] = df['text_meaning'].apply(find_political_context)
    print("Keyword search complete.")
    print(df['is_political_keyword'].value_counts())
else:
    print("Error: 'text_meaning' column not found in DataFrame.")


Keyword search complete.
is_political_keyword
0    215
1    115
Name: count, dtype: int64


In [15]:
df1=pd.read_csv('/kaggle/working/outputs/submission_inference_only.csv')

In [16]:
logo=pd.read_csv('/kaggle/input/logo-detection/logo scores.csv')
logo.head()
# Assuming both dataframes have the same index or are aligned by 'image_name'
# Merging is the safest way to ensure the confidence score aligns with the correct row in df.
merged_df = df.merge(logo[['Image_name', 'Confidence_score']], on='Image_name', how='left')

import numpy as np

# Apply the condition: if 'Confidence_score' > 0.88, set 'is_political_keyword' to 1.
# Note: Since the existing 'is_political_keyword' values are 0 or 1 (integers),
# we must ensure the new values are also treated as integers for consistency.

merged_df['is_political_keyword'] = np.where(
    # Condition: Confidence_score is greater than 0.88
    merged_df['Confidence_score'] > 0.88,

    # Value if True: Set to 1
    1,

    # Value if False: Keep the original value from the 'is_political_keyword' column
    merged_df['is_political_keyword']
)

# Now, update the original 'df' with the new column (this assumes 'df' is what you want to work with later)
df['is_political_keyword'] = merged_df['is_political_keyword']

# Update 'Target' in df1 to 'Political' ONLY where df has a 1
df1.loc[df['is_political_keyword'] == 1, 'Target'] = 'Political'

# Verify the changes
print(df1['Target'].value_counts())


Target
Political       182
NonPolitical    148
Name: count, dtype: int64


In [17]:
df1.head()

,Image_name,Target
0,test0001.jpg,Political
1,test0002.jpg,NonPolitical
2,test0003.jpg,Political
3,test0004.jpg,Political
4,test0005.jpg,NonPolitical


In [20]:
# ============================================================
# Face Recognition Override with InsightFace (buffalo_l)
# - Loads gallery NPZ (Name + 512D embedding)
# - Checks each meme image for known faces
# - Forces Target = "Political" if any known face appears
# - Writes final submission.csv
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from insightface.app import FaceAnalysis

# ------------------------------------------------------------
# 1) Load face embedding gallery from NPZ
#    Your file structure: arr_0 is (N, 2): [Name, embedding]
# ------------------------------------------------------------

# 🔴 CHANGE THIS to your real Kaggle input path
#   e.g. "/kaggle/input/face-embeddings/dataframe_buffalo_l_altogether2 (1).npz"
NPZ_PATH = "/kaggle/input/altogether-trimmed/dataframe_buffalo_l_altogether2 (1).npz"

npz = np.load(NPZ_PATH, allow_pickle=True)
print("NPZ keys:", npz.files)

xvalues = npz["arr_0"]   # shape (N, 2), dtype=object
col_name = npz["arr_1"]  # ['Name', 'Facial_Features'] (not strictly needed)

print("xvalues shape:", xvalues.shape, "dtype:", xvalues.dtype)
print("column names:", col_name)

# xvalues[i, 0] -> name (str)
# xvalues[i, 1] -> embedding array of shape (512,)

face_names = [str(row[0]) for row in xvalues]
emb_list   = [np.asarray(row[1], dtype=np.float32).reshape(-1) for row in xvalues]

face_emb_matrix = np.vstack(emb_list)  # (N, 512)
print(f"Loaded {len(face_names)} known faces.")
print("Embedding matrix shape:", face_emb_matrix.shape)

# Normalize gallery embeddings once (for cosine similarity)
face_emb_norms         = np.linalg.norm(face_emb_matrix, axis=1, keepdims=True) + 1e-9
face_emb_matrix_normed = face_emb_matrix / face_emb_norms

# ------------------------------------------------------------
# 2) Initialize InsightFace model (buffalo_l)
#    (Make sure you installed insightface + onnxruntime at top of notebook)
# ------------------------------------------------------------
faceapp = FaceAnalysis(
    name="buffalo_l",
    root="/kaggle/working/insightface_models",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
faceapp.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.5)
print("InsightFace (buffalo_l) initialized.")

# ------------------------------------------------------------
# 3) Helper: check if an image has any known political face
# ------------------------------------------------------------

# 🔴 CHANGE THIS if your test images live somewhere else
TEST_IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Test/Image"

def has_political_face_for_image(img_name, threshold=0.5):
    """
    Returns (has_face, matched_names_list) for a given image.
    has_face is True if any known face passes the cosine similarity threshold.
    """
    # img_name may be just "xxx.jpg" or full path
    if os.path.isabs(img_name) or os.path.exists(img_name):
        img_path = img_name
    else:
        img_path = os.path.join(TEST_IMG_DIR, img_name)

    img = cv2.imread(img_path)
    if img is None:
        # Image missing / unreadable
        return False, []

    faces = faceapp.get(img)
    if not faces:
        return False, []

    found_names = set()

    for f in faces:
        emb = f["embedding"].astype(np.float32).reshape(-1)
        norm = np.linalg.norm(emb) + 1e-9
        if norm == 0:
            continue
        emb_normed = emb / norm

        # cosine similarity with all gallery embeddings
        sims = face_emb_matrix_normed @ emb_normed  # (N,)
        match_idxs = np.where(sims > threshold)[0]

        for idx in match_idxs:
            found_names.add(face_names[idx])

    return (len(found_names) > 0), sorted(found_names)

# ------------------------------------------------------------
# 4) Apply face-based override on df1
#    Assumes df1 already exists with ["Image_name", "Target"]
# ------------------------------------------------------------

assert "Image_name" in df1.columns and "Target" in df1.columns, "df1 must have Image_name and Target columns"

df1["has_political_face"]   = False
df1["political_face_names"] = ""

for idx, row in tqdm(df1.iterrows(), total=len(df1), desc="Running face recognition override"):
    img_name = row["Image_name"]
    has_face, names_hit = has_political_face_for_image(img_name, threshold=0.5)

    if has_face:
        df1.at[idx, "has_political_face"]   = True
        df1.at[idx, "political_face_names"] = ";".join(names_hit)

# Force label to Political where any known face is present
df1.loc[df1["has_political_face"], "Target"] = "Political"

# ------------------------------------------------------------
# 5) Save final submission
# ------------------------------------------------------------
df1[["Image_name", "Target"]].to_csv("submission.csv", index=False)
print("submission.csv written with face-recognition override applied.")


NPZ keys: ['arr_0', 'arr_1']
xvalues shape: (324, 2) dtype: object
column names: ['Name' 'Facial_Features']
Loaded 324 known faces.
Embedding matrix shape: (324, 512)
download_path: /kaggle/working/insightface_models/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 73925.68KB/s] 


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /kaggle/working/insightface_models/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /kaggle/working/insightface_models/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /kaggle/working/insightface_models/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /kaggle/working/insightface_models/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /kaggle/working/insightface_models/models/buffalo_l/w600k_r50.onnx rec

Running face recognition override:   0%|          | 0/330 [00:00<?, ?it/s]

submission.csv written with face-recognition override applied.


In [ ]:
# df1.to_csv('submission.csv',index=False)